In [1]:
import os, pickle, shutil
import numpy as np
import pandas as pd
from recsys_utils import fix_text

PROC, OUT = '../data/processed', '../app/app_data'
os.makedirs(OUT, exist_ok=True)

split = pickle.load(open(f'{PROC}/split.pkl', 'rb'))
listings = pd.read_pickle(f'{PROC}/listings_clean.pkl').reset_index(drop=True)
feats = pd.read_pickle(f'{PROC}/item_features.pkl')
f = np.load('../models/als_factors.npz')
trainval, user_ids, pop = split['trainval'], split['user_ids'], split['pop_trainval']

listings['name'] = listings['name'].map(fix_text)
cols = ['listing_id', 'name', 'city', 'neighbourhood', 'room_type', 'price', 'bedrooms',
        'accommodates', 'latitude', 'longitude', 'number_of_reviews', 'review_scores_rating']
listings[cols].to_csv(f'{OUT}/listings_slim.csv.gz', index=False, compression='gzip')

active = np.where(pop > 0)[0].astype(np.int32)
demo = split['eval_users'].astype(np.int64)
hist_ptr, hist_idx = [0], []
for u in demo:
    h = trainval.indices[trainval.indptr[u]:trainval.indptr[u + 1]]
    hist_idx.extend(h.tolist()); hist_ptr.append(len(hist_idx))
test_item = np.array([next(iter(split['test_truth'][int(u)])) for u in demo], dtype=np.int32)

np.savez_compressed(f'{OUT}/als_demo.npz',
    item_idx=active,
    item_factors=f['item_factors'][active].astype(np.float16),
    demo_user_ids=user_ids[demo].astype(np.int64),
    user_factors=f['user_factors'][demo].astype(np.float16),
    hist_ptr=np.array(hist_ptr, dtype=np.int64),
    hist_idx=np.array(hist_idx, dtype=np.int32),
    test_item=test_item)

np.savez_compressed(f'{OUT}/item_features.npz', X=feats.values.astype(np.float16))

os.makedirs(f'{OUT}/results', exist_ok=True)
for name in ['summary_table', 'cold_start_table', 'hybrid_table', 'als_tuning', 'knn_tuning', 'ci_table']:
    p = f'{PROC}/{name}.csv'
    if os.path.exists(p):
        shutil.copy(p, f'{OUT}/results/{name}.csv')

for fn in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, fn)
    if os.path.isfile(p):
        print(f"{fn:22s} {os.path.getsize(p)/1e6:6.1f} MB")

als_demo.npz             25.8 MB
item_features.npz         2.2 MB
listings_slim.csv.gz     10.7 MB
